## Path configuration

In [1]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

Working directory: /export/usuarios01/agnavarr/MALDIAlign


## Imports

In [2]:
import pickle
import numpy as np
import pandas as pd

from src.config.loader import load_config
from src.data.io import load_pkl

In [3]:
cfg = load_config()

## Data loading

In [4]:
ecef = load_pkl("/export/data_ml4ds/bacteria_id/MALDIAlign_Alex/E-CEF.pkl")
kcef = load_pkl("/export/data_ml4ds/bacteria_id/MALDIAlign_Alex/K-CEF.pkl")
soxa = load_pkl("/export/data_ml4ds/bacteria_id/MALDIAlign_Alex/S-OXA.pkl")
kmulti = load_pkl(cfg["data"]["KLEBSIELLA_MULTIAMR"])
kmulti_marisma = load_pkl(cfg["data"]["MARISMA_KLEBSIELLA_MULTIAMR"])

In [5]:
ecef_data, ecef_label, ecef_meta, ecef_amr = ecef["data"], ecef["label"], ecef["meta"], ecef["amr"]
kcef_data, kcef_label, kcef_meta, kcef_amr = kcef["data"], kcef["label"], kcef["meta"], kcef["amr"]
soxa_data, soxa_label, soxa_meta, soxa_amr = soxa["data"], soxa["label"], soxa["meta"], soxa["amr"]
kmulti_data, kmulti_label, kmulti_meta, kmulti_amr, kmulti_list = kmulti["data"], kmulti["label"], kmulti["meta"], kmulti["amr"], kmulti["antibiotics"]
kmulti_meta = pd.DataFrame(list(kmulti_meta)) if isinstance(kmulti_meta, (list, np.ndarray)) else pd.DataFrame(kmulti_meta)
kmulti_marisma_data, kmulti_marisma_label, kmulti_marisma_meta, kmulti_marisma_amr, kmulti_marisma_list = kmulti_marisma["data"], kmulti_marisma["label"], kmulti_marisma["meta"], kmulti_marisma["amr"], kmulti_marisma["antibiotics"]

In [4]:
driams_dict = load_pkl(cfg["data"]["DRIAMS_FULL"])
marisma_dict = load_pkl(cfg["data"]["MARISMa_FULL"])
msumg_dict = load_pkl(cfg["data"]["MSUMG_FULL"])

species_to_keep = [
    "Klebsiella_Pneumoniae", "Escherichia_Coli", "Staphylococcus_Aureus",
    "Pseudomonas_Aeruginosa", "Enterococcus_Faecium", "Enterobacter_cloacae_complex"
]

mask_driams = np.isin(driams_dict["label"], species_to_keep)
data_driams = driams_dict["data"][mask_driams]
label_driams = driams_dict["label"][mask_driams]
meta_driams = pd.DataFrame.from_records(list(driams_dict["meta"]))[mask_driams].reset_index(drop=True)
amr_driams = driams_dict["amr"][mask_driams]
amr_list_driams = driams_dict["antibiotics"]

mask_marisma = np.isin(marisma_dict["label"], species_to_keep)
data_marisma = marisma_dict["data"][mask_marisma]
label_marisma = marisma_dict["label"][mask_marisma]
meta_marisma = pd.DataFrame.from_records(list(marisma_dict["meta"]))[mask_marisma].reset_index(drop=True)
meta_marisma.insert(
        loc=0,
        column="hospital",
        value="MARISMA"
    )
amr_marisma = marisma_dict["amr"][mask_marisma]
amr_list_marisma = marisma_dict["antibiotics"] 

mask_msumg = np.isin(msumg_dict["label"], species_to_keep)
data_msumg = msumg_dict["data"][mask_msumg]
label_msumg = msumg_dict["label"][mask_msumg]
meta_msumg = pd.DataFrame.from_records(list(msumg_dict["meta"]))[mask_msumg].reset_index(drop=True)
amr_msumg = msumg_dict["amr"][mask_msumg]
amr_list_msumg = msumg_dict["antibiotics"]
meta_msumg.insert(
        loc=0,
        column="hospital",
        value="MS-UMG"
    )

# Filtrar chrom-agar de MS-UMG
if "agar" in meta_msumg.columns:
    chrom_mask   = (meta_msumg["agar"] == "chrom").values
    keep_mask    = ~chrom_mask
    data_msumg   = data_msumg[keep_mask]
    label_msumg  = label_msumg[keep_mask]
    amr_msumg    = amr_msumg[keep_mask]
    meta_msumg   = meta_msumg[keep_mask].reset_index(drop=True)
    print(f"MS-UMG: eliminadas {chrom_mask.sum()} muestras chrom-agar. Quedan {keep_mask.sum()}")

MS-UMG: eliminadas 4437 muestras chrom-agar. Quedan 28290


## Data exploration

In [5]:
def analyze_filtered_amr_by_center(labels, amr_matrix, atb_list, meta_df, dataset_name):
    results = []
    centers = meta_df['hospital'].unique()
    
    for center in centers:
        center_mask = (meta_df['hospital'] == center).values
        
        c_labels = labels[center_mask]
        c_amr = amr_matrix[center_mask]
        
        unique_species = np.unique(c_labels)
        
        for species in unique_species:
            species_mask = (c_labels == species)
            
            for i, atb in enumerate(atb_list):
                atb_labels = c_amr[species_mask, i]
                valid_labels = atb_labels[~np.isnan(atb_labels)]
                
                if len(valid_labels) > 0:
                    n_total = len(valid_labels)
                    n_res = int(np.sum(valid_labels == 1))
                    n_sus = int(np.sum(valid_labels == 0))
                    prevalence = (n_res / n_total) * 100
                    
                    results.append({
                        "dataset": dataset_name,
                        "center": center,
                        "genus_species": species,
                        "antibiotic": atb,
                        "total_samples": n_total,
                        "R": n_res,
                        "S": n_sus,
                        "prevalence_R": round(prevalence, 2)
                    })
                
    return pd.DataFrame(results)

df_stats_driams = analyze_filtered_amr_by_center(
    label_driams, amr_driams, amr_list_driams, meta_driams, "DRIAMS"
)

df_stats_marisma = analyze_filtered_amr_by_center(
    label_marisma, amr_marisma, amr_list_marisma, meta_marisma, "MARISMa"
)

df_stats_msumg = analyze_filtered_amr_by_center(
    label_msumg, amr_msumg, amr_list_msumg, meta_msumg, "MSUMG"
)

df_final_prevalence = pd.concat([df_stats_driams, df_stats_marisma, df_stats_msumg], ignore_index=True)
print(df_final_prevalence.to_string(index=False))

dataset   center                genus_species                    antibiotic  total_samples     R     S  prevalence_R
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex   Amoxicillin-Clavulanic acid             95    95     0        100.00
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                      Amikacin             93     0    93          0.00
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                    Ampicillin             95    95     0        100.00
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                      Cefepime             95    10    85         10.53
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                     Cefoxitin             95    95     0        100.00
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                   Ceftazidime             95    22    73         23.16
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                   Ceftriaxone             95    22    73         23.16
 DRIAMS DRIAMS_B Enterobacter_cloacae_complex                   

In [6]:
# ============================================================
# NORMALIZAR NOMBRES DE ANTIBIÓTICOS
# ============================================================
name_map = {
    "Piperacillin/Tazobactam":       "Piperacillin-Tazobactam",
    "Amoxicillin/Clavulanic acid":   "Amoxicillin-Clavulanic acid",
    "Ceftazidime/Avibactam":         "Ceftazidime-Avibactam",
    "Ceftolozane/Tazobactam":        "Ceftolozane-Tazobactam",
    "Ampicillin/Sulbactam":          "Ampicillin+Sulbactam",
    "Trimethoprim/Sulfamethoxazole": "Cotrimoxazole",
    "Cotrimoxazol":                  "Cotrimoxazole",
    "Cefotaxim":                     "Cefotaxime",
}

df_final_prevalence["antibiotic"] = df_final_prevalence["antibiotic"].replace(name_map)

# Ahora re-agregar por si quedaron duplicados tras el renombrado
df_final_prevalence = (
    df_final_prevalence
    .groupby(["dataset", "center", "genus_species", "antibiotic"], as_index=False)
    .agg({
        "total_samples": "sum",
        "R": "sum",
        "S": "sum",
    })
)
df_final_prevalence["prevalence_R"] = (
    df_final_prevalence["R"] / df_final_prevalence["total_samples"] * 100
).round(2)

print("Nombres normalizados.")
print(f"Total rows: {len(df_final_prevalence)}")

Nombres normalizados.
Total rows: 738


In [7]:
train_centers = ["DRIAMS_A", "DRIAMS_B", "DRIAMS_C", "MARISMA"]
ood_centers   = ["MS-UMG"]
all_centers   = train_centers + ood_centers

species_list = sorted(df_final_prevalence["genus_species"].unique())

for species in species_list:
    df_sp = df_final_prevalence[
        (df_final_prevalence["genus_species"] == species) &
        (df_final_prevalence["center"].isin(all_centers))
    ].sort_values(["antibiotic", "center"])

    print("\n" + "="*76)
    print(f" {species}  (n_rows={len(df_sp)})")
    print("="*76)
    print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>8}")
    print("-"*76)

    prev_atb = None
    for _, row in df_sp.iterrows():
        if row["antibiotic"] != prev_atb:
            if prev_atb is not None:
                print()
            prev_atb = row["antibiotic"]
        print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>7.1f}%")


 Enterobacter_cloacae_complex  (n_rows=112)
Antibiotic                     Center              S        R    Total       %R
----------------------------------------------------------------------------
Amikacin                       DRIAMS_A         1452       19     1471     1.3%
Amikacin                       DRIAMS_B           93        0       93     0.0%
Amikacin                       DRIAMS_C          132        0      132     0.0%
Amikacin                       MARISMA            96        3       99     3.0%
Amikacin                       MS-UMG              0        1        1   100.0%

Amoxicillin-Clavulanic acid    DRIAMS_A            0     1474     1474   100.0%
Amoxicillin-Clavulanic acid    DRIAMS_B            0       95       95   100.0%
Amoxicillin-Clavulanic acid    DRIAMS_C            0      196      196   100.0%
Amoxicillin-Clavulanic acid    MARISMA             0       99       99   100.0%

Ampicillin                     DRIAMS_B            0       95       95   100

In [10]:
def select_final_amr_pairs(
    df,
    min_r_ood=5,
    min_s_ood=5,
    min_r_train_total=20,
    min_s_train_total=20,
    min_train_centers=2,
    imbalance_threshold=0.05
):
    ood_centers = ['MS-UMG']
    train_centers = ['DRIAMS_A', 'DRIAMS_B', 'DRIAMS_C', 'MARISMA']

    def is_valid_pair(group):
        ood_data = group[group['center'].isin(ood_centers)]

        if len(ood_data['center'].unique()) < len(ood_centers):
            return False

        for center in ood_centers:
            sub = ood_data[ood_data['center'] == center]
            if len(sub) == 0:
                return False

            R = sub['R'].values[0]
            S = sub['S'].values[0]

            if R < min_r_ood or S < min_s_ood:
                return False

        train_data = group[group['center'].isin(train_centers)]

        R_total = train_data['R'].sum()
        S_total = train_data['S'].sum()

        if R_total < min_r_train_total or S_total < min_s_train_total:
            return False

        ratio = R_total / (R_total + S_total)
        if ratio < imbalance_threshold or ratio > (1 - imbalance_threshold):
            return False

        informative_train = train_data[(train_data['R'] > 0) & (train_data['S'] > 0)]

        if len(informative_train['center'].unique()) < min_train_centers:
            return False

        return True

    mask = df.groupby(['genus_species', 'antibiotic']).apply(is_valid_pair)
    valid_indices = mask[mask].index.tolist()

    df_selection = (
        df.set_index(['genus_species', 'antibiotic'])
          .loc[valid_indices]
          .reset_index()
    )

    return df_selection

df_winners = select_final_amr_pairs(df_final_prevalence)

final_list = (
    df_winners[['genus_species', 'antibiotic']]
    .drop_duplicates()
    .sort_values(by='genus_species')
)

print(f"Se han seleccionado {len(final_list)} pares bicho-antibiótico.")
print("\n--- LISTA FINAL PARA EL PAPER ---")
print(final_list.to_string(index=False))

Se han seleccionado 31 pares bicho-antibiótico.

--- LISTA FINAL PARA EL PAPER ---
               genus_species              antibiotic
Enterobacter_cloacae_complex             Ceftazidime
Enterobacter_cloacae_complex             Ceftriaxone
Enterobacter_cloacae_complex           Ciprofloxacin
Enterobacter_cloacae_complex              Fosfomycin
Enterobacter_cloacae_complex Piperacillin-Tazobactam
        Enterococcus_Faecium              Ampicillin
        Enterococcus_Faecium                Imipenem
        Enterococcus_Faecium             Teicoplanin
        Enterococcus_Faecium              Vancomycin
            Escherichia_Coli Piperacillin-Tazobactam
            Escherichia_Coli              Fosfomycin
            Escherichia_Coli           Cotrimoxazole
            Escherichia_Coli              Gentamicin
            Escherichia_Coli             Ceftriaxone
            Escherichia_Coli             Ceftazidime
            Escherichia_Coli              Ampicillin
            Esch

In [11]:
def select_final_amr_pairs_v2(
    df,
    min_r_ood=10,
    min_s_ood=10,
    min_r_train_total=50,
    min_s_train_total=50,
    min_train_centers=2,
):
    ood_centers = ['MS-UMG']
    train_centers = ['DRIAMS_A', 'DRIAMS_B', 'DRIAMS_C', 'MARISMA']

    def is_valid_pair(group):
        # OOD check
        ood_data = group[group['center'].isin(ood_centers)]
        if len(ood_data['center'].unique()) < len(ood_centers):
            return False
        for center in ood_centers:
            sub = ood_data[ood_data['center'] == center]
            if len(sub) == 0:
                return False
            if sub['R'].values[0] < min_r_ood or sub['S'].values[0] < min_s_ood:
                return False

        # Train check
        train_data = group[group['center'].isin(train_centers)]
        if train_data['R'].sum() < min_r_train_total:
            return False
        if train_data['S'].sum() < min_s_train_total:
            return False

        # Al menos 2 centros con R>0 y S>0
        informative_train = train_data[(train_data['R'] > 0) & (train_data['S'] > 0)]
        if len(informative_train['center'].unique()) < min_train_centers:
            return False

        return True

    mask = df.groupby(['genus_species', 'antibiotic']).apply(is_valid_pair)
    valid_indices = mask[mask].index.tolist()

    df_selection = (
        df.set_index(['genus_species', 'antibiotic'])
          .loc[valid_indices]
          .reset_index()
    )
    return df_selection


df_winners = select_final_amr_pairs_v2(df_final_prevalence)
final_list = (
    df_winners[['genus_species', 'antibiotic']]
    .drop_duplicates()
    .sort_values(by='genus_species')
)
print(f"Se han seleccionado {len(final_list)} pares bicho-antibiótico.")
print("\n--- LISTA FINAL PARA EL PAPER ---")
print(final_list.to_string(index=False))

Se han seleccionado 36 pares bicho-antibiótico.

--- LISTA FINAL PARA EL PAPER ---
               genus_species              antibiotic
Enterobacter_cloacae_complex             Ceftazidime
Enterobacter_cloacae_complex             Ceftriaxone
Enterobacter_cloacae_complex           Ciprofloxacin
Enterobacter_cloacae_complex              Fosfomycin
Enterobacter_cloacae_complex Piperacillin-Tazobactam
        Enterococcus_Faecium              Ampicillin
        Enterococcus_Faecium                Imipenem
        Enterococcus_Faecium             Teicoplanin
        Enterococcus_Faecium              Vancomycin
            Escherichia_Coli Piperacillin-Tazobactam
            Escherichia_Coli              Gentamicin
            Escherichia_Coli              Fosfomycin
            Escherichia_Coli           Cotrimoxazole
            Escherichia_Coli             Ceftazidime
            Escherichia_Coli             Ceftriaxone
            Escherichia_Coli              Ampicillin
            Esch

In [12]:
# KLEBSIELLA
train_centers = ["DRIAMS_A", "DRIAMS_B", "DRIAMS_C", "MARISMA"]
ood_centers   = ["MS-UMG"]

antibiotics_interest = [
    "Imipenem", 
    "Meropenem",          
    "Ertapenem",        
    "Ceftazidime",    
    "Ceftriaxone"
    "Piperacillin-Tazobactam",  
    "Ciprofloxacin",           
]

df_kleb_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Klebsiella_Pneumoniae") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_kleb_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_kleb_check[
                (df_kleb_check["antibiotic"] == prev_atb) &
                (df_kleb_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_kleb_check[
                (df_kleb_check["antibiotic"] == prev_atb) &
                (df_kleb_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_kleb_check[
                (df_kleb_check["antibiotic"] == prev_atb) &
                (df_kleb_check["center"].isin(train_centers)) &
                (df_kleb_check["R"] > 0) & (df_kleb_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

# Último antibiótico
train_r = df_kleb_check[
    (df_kleb_check["antibiotic"] == prev_atb) &
    (df_kleb_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_kleb_check[
    (df_kleb_check["antibiotic"] == prev_atb) &
    (df_kleb_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_kleb_check[
    (df_kleb_check["antibiotic"] == prev_atb) &
    (df_kleb_check["center"].isin(train_centers)) &
    (df_kleb_check["R"] > 0) & (df_kleb_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Ceftazidime                    DRIAMS_A         2402      430     2832   15.2%
Ceftazidime                    DRIAMS_B          134       18      152   11.8%
Ceftazidime                    DRIAMS_C          308       57      365   15.6%
Ceftazidime                    MARISMA         11809     1292    13101    9.9%
Ceftazidime                    MS-UMG           2713      456     3169   14.4%
  → Train R=1797, OOD R=456, centros informativos=4  Ok

Ciprofloxacin                  DRIAMS_A         2325      513     2838   18.1%
Ciprofloxacin                  DRIAMS_B          130       22      152   14.5%
Ciprofloxacin                  DRIAMS_C          318       48      366   13.1%
Ciprofloxacin                  MARISMA         11488     4356    15844   27.5%
Ciprofloxacin                  MS-UMG           2782      387     3169   12.

In [13]:
# E. COLI
antibiotics_interest_ecoli = [
    "Imipenem",                 
    "Meropenem",           
    "Ertapenem",              
    "Ceftazidime",       
    "Ceftriaxone",             
    "Piperacillin-Tazobactam", 
    "Ciprofloxacin",      
    "Ampicillin",   
]

df_ecoli_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Escherichia_Coli") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest_ecoli))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_ecoli_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_ecoli_check[
                (df_ecoli_check["antibiotic"] == prev_atb) &
                (df_ecoli_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_ecoli_check[
                (df_ecoli_check["antibiotic"] == prev_atb) &
                (df_ecoli_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_ecoli_check[
                (df_ecoli_check["antibiotic"] == prev_atb) &
                (df_ecoli_check["center"].isin(train_centers)) &
                (df_ecoli_check["R"] > 0) & (df_ecoli_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

# Último antibiótico
train_r = df_ecoli_check[
    (df_ecoli_check["antibiotic"] == prev_atb) &
    (df_ecoli_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_ecoli_check[
    (df_ecoli_check["antibiotic"] == prev_atb) &
    (df_ecoli_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_ecoli_check[
    (df_ecoli_check["antibiotic"] == prev_atb) &
    (df_ecoli_check["center"].isin(train_centers)) &
    (df_ecoli_check["R"] > 0) & (df_ecoli_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Ampicillin                     DRIAMS_B           77      136      213   63.9%
Ampicillin                     DRIAMS_C          340      544      884   61.5%
Ampicillin                     MARISMA           532     1025     1557   65.8%
Ampicillin                     MS-UMG           5488     5546    11034   50.3%
  → Train R=1705, OOD R=5546, centros informativos=3  Ok

Ceftazidime                    DRIAMS_A         3951      871     4822   18.1%
Ceftazidime                    DRIAMS_B          168       45      213   21.1%
Ceftazidime                    DRIAMS_C          769      146      915   16.0%
Ceftazidime                    MARISMA          1247      338     1585   21.3%
Ceftazidime                    MS-UMG           9576     1459    11035   13.2%
  → Train R=1400, OOD R=1459, centros informativos=4  Ok

Ceftriaxone      

In [14]:
# ENTEROBACTER
antibiotics_interest_entero = [
    "Imipenem",
    "Meropenem",
    "Ertapenem",
    "Ceftazidime",
    "Ceftriaxone",
    "Piperacillin-Tazobactam",
    "Ciprofloxacin",
]

df_entero_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Enterobacter_cloacae_complex") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest_entero))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_entero_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_entero_check[
                (df_entero_check["antibiotic"] == prev_atb) &
                (df_entero_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_entero_check[
                (df_entero_check["antibiotic"] == prev_atb) &
                (df_entero_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_entero_check[
                (df_entero_check["antibiotic"] == prev_atb) &
                (df_entero_check["center"].isin(train_centers)) &
                (df_entero_check["R"] > 0) & (df_entero_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

train_r = df_entero_check[
    (df_entero_check["antibiotic"] == prev_atb) &
    (df_entero_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_entero_check[
    (df_entero_check["antibiotic"] == prev_atb) &
    (df_entero_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_entero_check[
    (df_entero_check["antibiotic"] == prev_atb) &
    (df_entero_check["center"].isin(train_centers)) &
    (df_entero_check["R"] > 0) & (df_entero_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Ceftazidime                    DRIAMS_A         1033      415     1448   28.7%
Ceftazidime                    DRIAMS_B           73       22       95   23.2%
Ceftazidime                    DRIAMS_C          146       50      196   25.5%
Ceftazidime                    MARISMA            66       31       97   32.0%
Ceftazidime                    MS-UMG           1165      443     1608   27.6%
  → Train R=518, OOD R=443, centros informativos=4  Ok

Ceftriaxone                    DRIAMS_A         1031      432     1463   29.5%
Ceftriaxone                    DRIAMS_B           73       22       95   23.2%
Ceftriaxone                    DRIAMS_C          144       52      196   26.5%
Ceftriaxone                    MS-UMG            873      284     1157   24.6%
  → Train R=506, OOD R=284, centros informativos=3  Ok

Ciprofloxacin        

In [15]:
# PSEUDOMONAS
antibiotics_interest_pseudo = [
    "Imipenem",     
    "Meropenem",          
    "Ceftazidime",   
    "Piperacillin-Tazobactam",
    "Ciprofloxacin",  
    "Amikacin",          
    "Ceftazidime-Avibactam",   
    "Ceftolozane-Tazobactam", 
]

df_pseudo_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Pseudomonas_Aeruginosa") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest_pseudo))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_pseudo_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_pseudo_check[
                (df_pseudo_check["antibiotic"] == prev_atb) &
                (df_pseudo_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_pseudo_check[
                (df_pseudo_check["antibiotic"] == prev_atb) &
                (df_pseudo_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_pseudo_check[
                (df_pseudo_check["antibiotic"] == prev_atb) &
                (df_pseudo_check["center"].isin(train_centers)) &
                (df_pseudo_check["R"] > 0) & (df_pseudo_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

train_r = df_pseudo_check[
    (df_pseudo_check["antibiotic"] == prev_atb) &
    (df_pseudo_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_pseudo_check[
    (df_pseudo_check["antibiotic"] == prev_atb) &
    (df_pseudo_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_pseudo_check[
    (df_pseudo_check["antibiotic"] == prev_atb) &
    (df_pseudo_check["center"].isin(train_centers)) &
    (df_pseudo_check["R"] > 0) & (df_pseudo_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Amikacin                       DRIAMS_A         2208      164     2372    6.9%
Amikacin                       DRIAMS_B          126        9      135    6.7%
Amikacin                       DRIAMS_C          338       19      357    5.3%
Amikacin                       MARISMA          1458       75     1533    4.9%
Amikacin                       MS-UMG           1304     2125     3429   62.0%
  → Train R=267, OOD R=2125, centros informativos=4  Ok

Ceftazidime                    DRIAMS_A         2227      232     2459    9.4%
Ceftazidime                    DRIAMS_B          136        2      138    1.4%
Ceftazidime                    DRIAMS_C          325       32      357    9.0%
Ceftazidime                    MARISMA             0     1509     1509  100.0%
Ceftazidime                    MS-UMG              0     3644     3644  100.

In [17]:
#STAPHYLOCOCCUS
antibiotics_interest_staph = [
    "Oxacillin",     
    "Ciprofloxacin", 
    # "Clindamycin",   
    # "Erythromycin",
    # "Tetracycline",
    # "Gentamicin", 
    # "Mupirocin",
]

df_staph_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Staphylococcus_Aureus") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest_staph))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_staph_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_staph_check[
                (df_staph_check["antibiotic"] == prev_atb) &
                (df_staph_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_staph_check[
                (df_staph_check["antibiotic"] == prev_atb) &
                (df_staph_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_staph_check[
                (df_staph_check["antibiotic"] == prev_atb) &
                (df_staph_check["center"].isin(train_centers)) &
                (df_staph_check["R"] > 0) & (df_staph_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

train_r = df_staph_check[
    (df_staph_check["antibiotic"] == prev_atb) &
    (df_staph_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_staph_check[
    (df_staph_check["antibiotic"] == prev_atb) &
    (df_staph_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_staph_check[
    (df_staph_check["antibiotic"] == prev_atb) &
    (df_staph_check["center"].isin(train_centers)) &
    (df_staph_check["R"] > 0) & (df_staph_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Ciprofloxacin                  DRIAMS_A         3141      616     3757   16.4%
Ciprofloxacin                  DRIAMS_B          322       26      348    7.5%
Ciprofloxacin                  DRIAMS_C          705       33      738    4.5%
Ciprofloxacin                  MARISMA             0     1689     1689  100.0%
  → Train R=2364, OOD R=0, centros informativos=3  Not ok

Oxacillin                      DRIAMS_A         3064      726     3790   19.2%
Oxacillin                      DRIAMS_B          325       21      346    6.1%
Oxacillin                      DRIAMS_C          697       41      738    5.6%
Oxacillin                      MARISMA          1882      458     2340   19.6%
Oxacillin                      MS-UMG           6134      559     6693    8.3%
  → Train R=1246, OOD R=559, centros informativos=4  Ok


In [19]:
# ENTEROCOCCUS
antibiotics_interest_entero_fae = [
    "Vancomycin", 
    "Linezolid",    
    # "Teicoplanin",  
    # "Daptomycin", 
]

df_entero_fae_check = df_final_prevalence[
    (df_final_prevalence["genus_species"] == "Enterococcus_Faecium") &
    (df_final_prevalence["center"].isin(train_centers + ood_centers)) &
    (df_final_prevalence["antibiotic"].isin(antibiotics_interest_entero_fae))
].sort_values(["antibiotic", "center"])

print(f"{'Antibiotic':<30} {'Center':<12} {'S':>8} {'R':>8} {'Total':>8} {'%R':>7}")
print("-"*76)

prev_atb = None
for _, row in df_entero_fae_check.iterrows():
    if row["antibiotic"] != prev_atb:
        if prev_atb is not None:
            train_r = df_entero_fae_check[
                (df_entero_fae_check["antibiotic"] == prev_atb) &
                (df_entero_fae_check["center"].isin(train_centers))
            ]["R"].sum()
            ood_r = df_entero_fae_check[
                (df_entero_fae_check["antibiotic"] == prev_atb) &
                (df_entero_fae_check["center"].isin(ood_centers))
            ]["R"].sum()
            n_informative = df_entero_fae_check[
                (df_entero_fae_check["antibiotic"] == prev_atb) &
                (df_entero_fae_check["center"].isin(train_centers)) &
                (df_entero_fae_check["R"] > 0) & (df_entero_fae_check["S"] > 0)
            ]["center"].nunique()
            status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
            print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")
            print()
        prev_atb = row["antibiotic"]
    print(f"{row['antibiotic']:<30} {row['center']:<12} {row['S']:>8} {row['R']:>8} {row['total_samples']:>8} {row['prevalence_R']:>6.1f}%")

train_r = df_entero_fae_check[
    (df_entero_fae_check["antibiotic"] == prev_atb) &
    (df_entero_fae_check["center"].isin(train_centers))
]["R"].sum()
ood_r = df_entero_fae_check[
    (df_entero_fae_check["antibiotic"] == prev_atb) &
    (df_entero_fae_check["center"].isin(ood_centers))
]["R"].sum()
n_informative = df_entero_fae_check[
    (df_entero_fae_check["antibiotic"] == prev_atb) &
    (df_entero_fae_check["center"].isin(train_centers)) &
    (df_entero_fae_check["R"] > 0) & (df_entero_fae_check["S"] > 0)
]["center"].nunique()
status = "Ok" if (train_r >= 50 and ood_r >= 10 and n_informative >= 2) else "Not ok"
print(f"  → Train R={train_r}, OOD R={ood_r}, centros informativos={n_informative}  {status}")

Antibiotic                     Center              S        R    Total      %R
----------------------------------------------------------------------------
Linezolid                      DRIAMS_A         1122        2     1124    0.2%
Linezolid                      DRIAMS_B           41        0       41    0.0%
Linezolid                      DRIAMS_C           90        3       93    3.2%
Linezolid                      MARISMA           879       21      900    2.3%
Linezolid                      MS-UMG           2051       42     2093    2.0%
  → Train R=26, OOD R=42, centros informativos=3  Not ok

Vancomycin                     DRIAMS_A         1087       96     1183    8.1%
Vancomycin                     DRIAMS_B           28       13       41   31.7%
Vancomycin                     DRIAMS_C           89        4       93    4.3%
Vancomycin                     MARISMA           808       95      903   10.5%
Vancomycin                     MS-UMG           1476      637     2113   30

In [11]:
golden_pairs = set(zip(df_winners['genus_species'], df_winners['antibiotic']))

# ============================================================
# NORMALIZAR TAMBIÉN LAS LISTAS DE ANTIBIÓTICOS DE LOS DATASETS
# ============================================================
name_map = {
    "Piperacillin/Tazobactam":       "Piperacillin-Tazobactam",
    "Amoxicillin/Clavulanic acid":   "Amoxicillin-Clavulanic acid",
    "Ceftazidime/Avibactam":         "Ceftazidime-Avibactam",
    "Ceftolozane/Tazobactam":        "Ceftolozane-Tazobactam",
    "Ampicillin/Sulbactam":          "Ampicillin+Sulbactam",
    "Trimethoprim/Sulfamethoxazole": "Cotrimoxazole",
    "Cotrimoxazol":                  "Cotrimoxazole",
    "Cefotaxim":                     "Cefotaxime",
}

amr_list_driams_norm  = [name_map.get(a, a) for a in amr_list_driams]
amr_list_marisma_norm = [name_map.get(a, a) for a in amr_list_marisma]
amr_list_msumg_norm   = [name_map.get(a, a) for a in amr_list_msumg]

# Ahora usa las listas normalizadas en el loop
datasets = {
    "DRIAMS":  (label_driams,  amr_driams,  amr_list_driams_norm,  meta_driams),
    "MARISMA": (label_marisma, amr_marisma, amr_list_marisma_norm, meta_marisma),
    "MS-UMG":  (label_msumg,  amr_msumg,   amr_list_msumg_norm,   meta_msumg),
}

print(f"\n{'='*110}")
print(f"{'PROPORCIONES DE LOS PARES SELECCIONADOS POR EL FILTRO AUTOMÁTICO':^110}")
print(f"{'='*110}")
print(f"{'Especie':<35} {'Antibiótico':<30} {'Centro':<15} {'S':>6} {'R':>6} {'Total':>8} {'Prev_%':>8}")
print(f"{'-'*110}")

for species, atb in sorted(golden_pairs):
    for dataset_name, (labels, amr, atb_list, meta) in datasets.items():

        if atb not in list(atb_list):
            continue

        sp_mask = (labels == species)
        if sp_mask.sum() == 0:
            continue

        j     = list(atb_list).index(atb)
        y     = amr[sp_mask, j]
        valid = ~np.isnan(y)
        if valid.sum() == 0:
            continue

        if dataset_name == "DRIAMS":
            meta_sp = meta[sp_mask].reset_index(drop=True)
            for center in sorted(meta_sp["hospital"].unique()):
                c_mask = (meta_sp["hospital"] == center).values
                y_c    = y[c_mask]
                v_c    = ~np.isnan(y_c)
                n_r_c  = int((y_c[v_c] == 1).sum())
                n_s_c  = int((y_c[v_c] == 0).sum())
                t_c    = n_r_c + n_s_c
                if t_c == 0:
                    continue
                prev = 100 * n_r_c / t_c
                print(f"{species:<35} {atb:<30} {center:<15} {n_s_c:>6} {n_r_c:>6} {t_c:>8} {prev:>8.1f}")
        else:
            n_r   = int((y[valid] == 1).sum())
            n_s   = int((y[valid] == 0).sum())
            total = n_r + n_s
            if total == 0:
                continue
            prev = 100 * n_r / total
            print(f"{species:<35} {atb:<30} {dataset_name:<15} {n_s:>6} {n_r:>6} {total:>8} {prev:>8.1f}")

    print(f"{'-'*110}")

print(f"{'='*110}")


                       PROPORCIONES DE LOS PARES SELECCIONADOS POR EL FILTRO AUTOMÁTICO                       
Especie                             Antibiótico                    Centro               S      R    Total   Prev_%
--------------------------------------------------------------------------------------------------------------
Enterobacter_cloacae_complex        Ceftazidime                    DRIAMS_A          1033    415     1448     28.7
Enterobacter_cloacae_complex        Ceftazidime                    DRIAMS_B            73     22       95     23.2
Enterobacter_cloacae_complex        Ceftazidime                    DRIAMS_C           146     50      196     25.5
Enterobacter_cloacae_complex        Ceftazidime                    DRIAMS_D           342     95      437     21.7
Enterobacter_cloacae_complex        Ceftazidime                    MARISMA             66     31       97     32.0
Enterobacter_cloacae_complex        Ceftazidime                    MS-UMG            11

In [13]:
def save_pkl(data, path):
    with open(path, 'wb') as f:
        pickle.dump(data, f)

def prepare_meta(data_dict, hospital_name=None):
    df_meta = pd.DataFrame(data_dict["meta"])
    
    if hospital_name and "hospital" not in df_meta.columns:
        df_meta.insert(loc=0, column="hospital", value=hospital_name)
    
    data_dict["meta"] = df_meta
    return data_dict


In [14]:
print("Preparando metadata...")
driams_dict = prepare_meta(driams_dict) 
marisma_dict = prepare_meta(marisma_dict, "MARISMA")
msumg_dict = prepare_meta(msumg_dict, "MS-UMG")

def process_and_filter(data, labels, meta, amr, atb_list_orig, species_list, target_atb_list, golden_pairs):
    species_mask = np.isin(labels, species_list)

    f_data   = data[species_mask]
    f_labels = labels[species_mask]
    f_meta   = meta[species_mask].reset_index(drop=True).to_dict('records')
    f_amr    = amr[species_mask]

    new_amr = np.full((len(f_labels), len(target_atb_list)), np.nan)
    for i, atb in enumerate(target_atb_list):
        if atb in atb_list_orig:
            old_idx  = list(atb_list_orig).index(atb)
            col_data = f_amr[:, old_idx]
            for j, sp in enumerate(f_labels):
                if (sp, atb) in golden_pairs:
                    new_amr[j, i] = col_data[j]

    return {"data": f_data, "label": f_labels, "meta": f_meta, "amr": new_amr}

print("Filtrando por species_to_keep y Golden Pairs...")
# ============================================================
# NORMALIZAR LISTAS DE ANTIBIÓTICOS
# ============================================================
name_map = {
    "Piperacillin/Tazobactam":       "Piperacillin-Tazobactam",
    "Amoxicillin/Clavulanic acid":   "Amoxicillin-Clavulanic acid",
    "Ceftazidime/Avibactam":         "Ceftazidime-Avibactam",
    "Ceftolozane/Tazobactam":        "Ceftolozane-Tazobactam",
    "Ampicillin/Sulbactam":          "Ampicillin+Sulbactam",
    "Trimethoprim/Sulfamethoxazole": "Cotrimoxazole",
    "Cotrimoxazol":                  "Cotrimoxazole",
    "Cefotaxim":                     "Cefotaxime",
}

amr_list_driams_norm  = [name_map.get(a, a) for a in amr_list_driams]
amr_list_marisma_norm = [name_map.get(a, a) for a in amr_list_marisma]
amr_list_msumg_norm   = [name_map.get(a, a) for a in amr_list_msumg]

# ============================================================
# PROCESAR Y FILTRAR
# ============================================================
target_antibiotics   = sorted(df_winners['antibiotic'].unique().tolist())
golden_pairs_set     = set(zip(df_winners['genus_species'], df_winners['antibiotic']))

d_f  = process_and_filter(data_driams,  label_driams,  meta_driams,  amr_driams,  amr_list_driams_norm,  species_to_keep, target_antibiotics, golden_pairs_set)
m_f  = process_and_filter(data_marisma, label_marisma, meta_marisma, amr_marisma, amr_list_marisma_norm, species_to_keep, target_antibiotics, golden_pairs_set)
ms_f = process_and_filter(data_msumg,   label_msumg,   meta_msumg,   amr_msumg,   amr_list_msumg_norm,   species_to_keep, target_antibiotics, golden_pairs_set)

print("Generando dataset global...")
global_dict = {
    "data":        np.vstack([d_f["data"], m_f["data"], ms_f["data"]]),
    "label":       np.concatenate([d_f["label"], m_f["label"], ms_f["label"]]),
    "amr":         np.vstack([d_f["amr"], m_f["amr"], ms_f["amr"]]),
    "meta":        d_f["meta"] + m_f["meta"] + ms_f["meta"],
    "antibiotics": target_antibiotics
}

save_pkl(global_dict, "/export/usuarios01/agnavarr/MALDIAlign/amr_global_final_v3.pkl")
print(f"DONE. Dataset creado con {len(global_dict['label'])} muestras.")
print(f"Antibióticos ({len(target_antibiotics)}): {target_antibiotics}")

Preparando metadata...
Filtrando por species_to_keep y Golden Pairs...
Generando dataset global...
DONE. Dataset creado con 79207 muestras.
Antibióticos (19): ['Amikacin', 'Ampicillin', 'Ceftazidime', 'Ceftriaxone', 'Ciprofloxacin', 'Clindamycin', 'Cotrimoxazole', 'Erythromycin', 'Fosfomycin', 'Fusidic acid', 'Gentamicin', 'Imipenem', 'Meropenem', 'Mupirocin', 'Oxacillin', 'Piperacillin-Tazobactam', 'Teicoplanin', 'Tetracycline', 'Vancomycin']


In [15]:
def imprimir_distribucion_amr(data_dict):
    df_base = pd.DataFrame(data_dict["meta"])
    df_base["species"] = data_dict["label"]
    
    df_amr = pd.DataFrame(data_dict["amr"], columns=data_dict["antibiotics"])
    
    df_combined = pd.concat([df_base[["hospital", "species"]], df_amr], axis=1)
    df_long = df_combined.melt(
        id_vars=["hospital", "species"], 
        var_name="antibiotic", 
        value_name="amr_value"
    )
    
    df_long = df_long.dropna(subset=["amr_value"])
    
    stats = df_long.groupby(["species", "antibiotic", "hospital", "amr_value"]).size().unstack(fill_value=0)
    stats.columns = ["S", "R"]
    stats["Total"] = stats["S"] + stats["R"]
    stats["Prev_%"] = (stats["R"] / stats["Total"] * 100).round(1)
    
    pd.set_option('display.max_rows', None)      
    pd.set_option('display.max_columns', None)  
    pd.set_option('display.width', 1000)         
    pd.set_option('display.colheader_justify', 'center')
    
    final_df = stats.reset_index()
    
    print("\n" + "="*100)
    print(f"{'DISTRIBUCIÓN AMR POR ESPECIE, ANTIBIÓTICO Y CENTRO':^100}")
    print("="*100)
    print(final_df.to_string(index=False))
    print("="*100 + "\n")

global_dict = load_pkl("/export/usuarios01/agnavarr/MALDIAlign/amr_global_final.pkl")
imprimir_distribucion_amr(global_dict)


                         DISTRIBUCIÓN AMR POR ESPECIE, ANTIBIÓTICO Y CENTRO                         
          species                   antibiotic       hospital    S    R   Total  Prev_%
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_A  1033  415  1448    28.7 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_C   146   50   196    25.5 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_D   342   95   437    21.7 
Enterobacter_cloacae_complex             Ceftazidime  MARISMA    66   31    97    32.0 
Enterobacter_cloacae_complex             Ceftazidime   MS-UMG  1175  753  1928    39.1 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_A  1031  432  1463    29.5 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_C   144   52   196    26.5 
Enterobacter_cloac

In [16]:
imprimir_distribucion_amr(load_pkl("/export/usuarios01/agnavarr/MALDIAlign/amr_global_v2.pkl"))


                         DISTRIBUCIÓN AMR POR ESPECIE, ANTIBIÓTICO Y CENTRO                         
          species                   antibiotic       hospital    S    R   Total  Prev_%
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_A  1033  415  1448    28.7 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_C   146   50   196    25.5 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_D   342   95   437    21.7 
Enterobacter_cloacae_complex             Ceftazidime  MARISMA    66   31    97    32.0 
Enterobacter_cloacae_complex             Ceftazidime   MS-UMG  1165  443  1608    27.5 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_A  1031  432  1463    29.5 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_C   144   52   196    26.5 
Enterobacter_cloac

In [17]:
imprimir_distribucion_amr(load_pkl("/export/usuarios01/agnavarr/MALDIAlign/amr_global_final_v3.pkl"))


                         DISTRIBUCIÓN AMR POR ESPECIE, ANTIBIÓTICO Y CENTRO                         
          species                   antibiotic       hospital    S    R   Total  Prev_%
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_A  1033  415  1448    28.7 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_C   146   50   196    25.5 
Enterobacter_cloacae_complex             Ceftazidime DRIAMS_D   342   95   437    21.7 
Enterobacter_cloacae_complex             Ceftazidime  MARISMA    66   31    97    32.0 
Enterobacter_cloacae_complex             Ceftazidime   MS-UMG  1165  443  1608    27.5 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_A  1031  432  1463    29.5 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_B    73   22    95    23.2 
Enterobacter_cloacae_complex             Ceftriaxone DRIAMS_C   144   52   196    26.5 
Enterobacter_cloac

In [18]:
# ============================================================
# VERIFICAR DISPONIBILIDAD DE MUESTRAS EN MS-UMG PARA FINETUNING
# ============================================================
print("\n" + "="*80)
print("DISPONIBILIDAD EN MS-UMG POR ESPECIE Y ANTIBIÓTICO")
print("="*80)

ms_meta = pd.DataFrame(ms_f["meta"])
ms_amr  = ms_f["amr"]
ms_labels = ms_f["label"]

print(f"\nTotal muestras MS-UMG: {len(ms_labels)}")
print(f"\n{'Especie':<35} {'Antibiótico':<30} {'S':>8} {'R':>8} {'NaN':>8} {'Total':>8}")
print("-"*90)

for j, atb in enumerate(target_antibiotics):
    for species in sorted(np.unique(ms_labels)):
        sp_mask = ms_labels == species
        if (species, atb) not in golden_pairs_set:
            continue

        col = ms_amr[sp_mask, j]
        n_r   = int((col == 1).sum())
        n_s   = int((col == 0).sum())
        n_nan = int(np.isnan(col).sum())
        total = n_r + n_s + n_nan

        # Flag si hay pocas muestras etiquetadas
        n_labeled = n_r + n_s
        flag = "⚠️" if n_labeled < 100 else "✅"

        print(f"{flag} {species:<33} {atb:<30} {n_s:>8} {n_r:>8} {n_nan:>8} {total:>8}")
    print()


DISPONIBILIDAD EN MS-UMG POR ESPECIE Y ANTIBIÓTICO

Total muestras MS-UMG: 28290

Especie                             Antibiótico                           S        R      NaN    Total
------------------------------------------------------------------------------------------
✅ Pseudomonas_Aeruginosa            Amikacin                           1304     2125      218     3647

✅ Enterococcus_Faecium              Ampicillin                          177     1934        4     2115
✅ Escherichia_Coli                  Ampicillin                         5488     5546        7    11041

✅ Enterobacter_cloacae_complex      Ceftazidime                        1165      443        2     1610
✅ Escherichia_Coli                  Ceftazidime                        9576     1459        6    11041
✅ Klebsiella_Pneumoniae             Ceftazidime                        2713      456        0     3169

✅ Enterobacter_cloacae_complex      Ceftriaxone                         873      284      453     1610